<a href="https://colab.research.google.com/github/AnirudhChandupatla/AI-ML-from-scratch/blob/master/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from sklearn.datasets import load_iris
iris = load_iris()

In [4]:
import pandas as pd
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species'] = df['species'].replace(to_replace= [0, 1, 2], value = ['setosa', 'versicolor', 'virginica'])
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [33]:
import numpy as np

class myPCA:
    def __init__(self, n_components=None, tol=1e-8, max_iter=1000):
        # Store user-specified parameters for later use
        self.n_components = n_components  # Number of principal components to keep
        self.tol = tol                   # Tolerance for convergence in QR algorithm
        self.max_iter = max_iter         # Maximum iterations for QR algorithm

    def fit(self, X):

        # If X is a DataFrame, extract column names and convert to numpy array
        # Why: To keep track of feature names for later interpretation.
        # If not done: Feature names will be lost, making results harder to interpret.
        if hasattr(X, 'columns'):
            self.feature_names_in_ = np.array(X.columns)
            X = X.values  # convert to numpy
        else:
            # If X is not a DataFrame, create generic feature names
            self.feature_names_in_ = np.array([f"x{i}" for i in range(X.shape[1])])

        # Store the number of samples and features
        # Why: Needed for later calculations (e.g., explained variance, singular values).
        # If not done: Later calculations may fail or be incorrect.
        self.n_samples_, self.n_features_ = X.shape

        # Set number of components to use
        # Why: Determines how many principal components to keep.
        # If not done: All features would be kept by default.
        if self.n_components is None:
            self.n_components_ = self.n_features_
        else:
            self.n_components_ = self.n_components

        # Step 1: Mean center the data
        # Why: PCA requires data to be centered at zero mean for each feature.
        # If not done: The first principal component may not capture the direction of maximum variance.
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_

        # Step 2: Compute the covariance matrix
        # Why: Covariance matrix captures the relationships between features.
        # If not done: Cannot find directions of maximum variance.
        cov_mat = (1 / (self.n_samples_-1)) * X_centered.T @ X_centered

        # Step 3: Get eigenvalues and eigenvectors using QR iteration
        # Why: Eigenvectors give principal directions, eigenvalues give variance explained.
        # If not done: Cannot determine principal components.
        eigen_values, eigen_vectors = self.qr_algorithm(cov_mat, self.max_iter)

        # Step 4: Sort eigenvalues and eigenvectors
        # Why: Principal components are ordered by explained variance (largest first).
        # If not done: Components may not be in order of importance.
        sorted_idx = np.argsort(eigen_values)[::-1]
        eigen_values = eigen_values[sorted_idx]
        eigen_vectors = eigen_vectors[:, sorted_idx]
        self.feature_names_in_ = self.feature_names_in_[sorted_idx]

        # Store the top n_components eigenvectors as principal axes
        # Why: These are the directions of maximum variance.
        # If not done: All components would be kept, possibly including noise.
        self.components_ = eigen_vectors[:, :self.n_components_].T

        # Store the explained variance for each component
        # Why: Useful for understanding how much variance each component explains.
        # If not done: Cannot assess the importance of each component.
        self.explained_variance_ = eigen_values[:self.n_components_]

        # Store the explained variance ratio
        # Why: Shows the proportion of variance explained by each component.
        # If not done: Harder to interpret the effectiveness of dimensionality reduction.
        self.explained_variance_ratio_ = self.explained_variance_ / eigen_values.sum()

        # Compute singular values from explained variance
        # Why: Useful for reconstructing the original data and for some downstream tasks.
        # If not done: Some applications may not work as expected.
        self.singular_values_ = np.sqrt(self.explained_variance_ * (self.n_samples_ - 1))

        # Estimate noise variance if not all components are kept
        # Why: Useful for probabilistic PCA and understanding residual variance.
        # If not done: Noise variance would be unknown.
        if self.n_components_ < self.n_features_:
            self.noise_variance_ = eigen_values[self.n_components_:].mean()
        else:
            self.noise_variance_ = 0.0

        # Return the fitted object for chaining
        return self

    def transform(self, X):
        # Center the input data using the mean from training data
        # Why: PCA projection assumes data is centered; otherwise, projections are incorrect.
        # If not done: The transformed data will be biased and not represent true principal components.
        X_centered = X - self.mean_
        # Project data onto principal components
        # Why: This reduces dimensionality and represents data in the new basis.
        # If not done: Data remains in original space, no dimensionality reduction.
        return np.dot(X_centered, self.components_.T)

    def fit_transform(self, X):
        # Fit the PCA model to the data
        # Why: Learns the principal components from the data.
        # If not done: Cannot transform data correctly.
        self.fit(X)
        # Transform the data using the learned components
        # Why: Projects data onto the principal component axes.
        # If not done: Data remains in original space.
        return self.transform(X)

    def inverse_transform(self, X_reduced):
        # Project reduced data back to original space
        # Why: Allows reconstruction of original data from reduced representation.
        # If not done: Cannot interpret reduced data in original feature space.
        return np.dot(X_reduced, self.components_) + self.mean_

    def qr_decomposition(self, A):
        '''
        Implementation of the Gram-Schmidt process for QR decomposition pseudo-code from the Strang's book
        ref: https://web.archive.org/web/20210506235252/http://mlwiki.org/index.php/Gram-Schmidt_Process
        Why: Used in the QR algorithm to decompose a matrix into orthogonal (Q) and upper triangular (R) matrices.
        If not done: Cannot perform QR algorithm for eigenvalue computation.
        '''
        m, n = A.shape
        Q = np.zeros((m, n))  # Initialize Q matrix
        R = np.zeros((n, n))  # Initialize R matrix

        for j in range(n):
            v = A[:, j]  # Take the j-th column of A

            for i in range(j):
                q = Q[:, i]            # i-th column of Q
                R[i, j] = q.dot(v)     # Project v onto q
                v = v - R[i, j] * q    # Subtract projection

            norm = np.linalg.norm(v)   # Compute norm of the vector
            Q[:, j] = v / norm         # Normalize to get orthogonal vector
            R[j, j] = norm             # Store norm in R

        return Q, R  # Return orthogonal and upper triangular matrices

    def qr_algorithm(self, A, max_iterations, tol=1e-10):
        """
        Find eigenvalues and eigenvectors using the QR Algorithm.
        Why: Used to compute the eigenvalues/vectors of the covariance matrix for PCA.
        If not done: Cannot determine principal components.
        """
        n = A.shape[0]
        Ak = A.copy()              # Copy of matrix to avoid modifying original
        Q_total = np.eye(n)        # Accumulate product of Q matrices for eigenvectors

        for iteration in range(max_iterations):
            Q, R = self.qr_decomposition(Ak)  # QR decomposition of Ak
            Ak = R @ Q                        # Form next Ak
            Q_total = Q_total @ Q              # Accumulate Q for eigenvectors

            # Check convergence: if off-diagonal elements are small enough, stop
            # Why: Indicates matrix is nearly diagonal (eigenvalues found).
            # If not done: May run unnecessary iterations or never stop.
            off_diagonal = Ak - np.diag(np.diag(Ak))
            if np.linalg.norm(off_diagonal) < tol:
                break

        eigenvalues = np.diag(Ak)   # Eigenvalues are on the diagonal
        eigenvectors = Q_total      # Columns of Q_total are eigenvectors
        return eigenvalues, eigenvectors


In [34]:
mypca = myPCA(n_components=4)
mypca.fit(df.drop('species', axis=1))
print(mypca.explained_variance_)
print(mypca.explained_variance_ratio_)
print(mypca.feature_names_in_)
print(sum(mypca.explained_variance_ratio_))
print(mypca.components_)
print(mypca.singular_values_)

[4.22824171 0.24267075 0.0782095  0.02383509]
[0.92461872 0.05306648 0.01710261 0.00521218]
['sepal length (cm)' 'sepal width (cm)' 'petal length (cm)'
 'petal width (cm)']
0.9999999999999999
[[ 0.36138659 -0.08452251  0.85667061  0.3582892 ]
 [ 0.65658877  0.73016143 -0.17337266 -0.07548102]
 [-0.58202985  0.59791083  0.07623608  0.54583143]
 [ 0.31548719 -0.3197231  -0.47983899  0.75365743]]
[25.09996044  6.01314738  3.41368064  1.88452351]


In [35]:
from sklearn.decomposition import PCA
pca = PCA(n_components=4)
pca.fit(df.drop('species', axis=1))
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)
print(pca.feature_names_in_)
print(sum(pca.explained_variance_ratio_))
print(pca.components_)
print(pca.singular_values_)

[4.22824171 0.24267075 0.0782095  0.02383509]
[0.92461872 0.05306648 0.01710261 0.00521218]
['sepal length (cm)' 'sepal width (cm)' 'petal length (cm)'
 'petal width (cm)']
0.9999999999999999
[[ 0.36138659 -0.08452251  0.85667061  0.3582892 ]
 [ 0.65658877  0.73016143 -0.17337266 -0.07548102]
 [-0.58202985  0.59791083  0.07623608  0.54583143]
 [ 0.31548719 -0.3197231  -0.47983899  0.75365743]]
[25.09996044  6.01314738  3.41368064  1.88452351]
